# AHỤIKE 03: benchmark AHỤIKE vs base N-ATLAS (vLLM, ~30-45 min), then live demo

Kaggle settings: **GPU T4 x2**, **Internet on**, Secret **HF_TOKEN** ticked for this notebook. Run with **Save Version > Save & Run All (Commit)**.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

In [ ]:
!git clone -q https://github.com/EgwuSamuel/ahuike.git /kaggle/working/ahuike || (cd /kaggle/working/ahuike && git pull -q)
%cd /kaggle/working/ahuike
!pip install -q vllm
!pip uninstall -y -q torchaudio
!python scripts/check_setup.py

In [ ]:
!python scripts/hub_sync.py pull data results outputs/ahuike-lora outputs/ablation-lora
!python scripts/build_sft.py

## Benchmark (base is re-run automatically if the prompt changed)

In [ ]:
!python scripts/run_eval.py --systems base base_fewshot ahuike=outputs/ahuike-lora ablation=outputs/ablation-lora
!python scripts/compute_metrics.py
!python scripts/hub_sync.py push results
!cat results/REPORT.md

## Live demo (restart the session first to free GPU memory, re-run the setup cells)
Opens a public Gradio link: use it for the video and for live user sessions.

In [ ]:
!pip install -q gradio
!python app/app.py --backend hf --adapter outputs/ahuike-lora --share

## Voice evaluation (after uploading recordings to voice/ and filling voice/manifest.csv)

In [ ]:
!python scripts/eval_voice.py --backend hf --adapter outputs/ahuike-lora